# NB_Silver_Order_Payments — Bronze to Silver Transformation (Proposed)

## Section 1 — Purpose and Business Context
- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.order_payments` → ADLS Bronze Parquet (`bronze/sales/order_payments/`).
- **Grain:** One row per payment attempt/instrument `(order_id, payment_sequential)` (`103,886` rows across `99,440` orders).
- **Observed Data Quality:** `0` nulls; `payment_type` includes `not_defined` (`3` rows with `0.00` value); `9` rows have `payment_value = 0.00`; `2` rows have `payment_installments = 0`. Preserve all verbatim.
- **Gold Usage:** Feeds `gold.fact_payments` (never collapse into `fact_order_items` without explicit grain separation).

> **WARNING:** **DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.** (`WRITE_ENABLED = False` by default.)

In [ ]:
# Section 2 — Configuration
from pyspark.sql import functions as F

WRITE_ENABLED = False  # SAFETY LOCK

BRONZE_PATH = "abfss://retail-lakehouse@stolistlakehouse01.dfs.core.windows.net/bronze/sales/order_payments/"
SILVER_TARGET = "silver.order_payments"
QUARANTINE_TARGET = "silver._quarantine_order_payments"
INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
VALID_PAYMENT_TYPES = ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]

In [ ]:
# Section 3 & 4 — Read Bronze Parquet (Exclude test_5rows outside subfolder)
df_bronze = spark.read.format("parquet").load(BRONZE_PATH)
df_bronze.printSchema()
bronze_count = df_bronze.count()
print(f"Bronze row count: {bronze_count}")

In [ ]:
# Section 5 — Data Quality Profiling
required_cols = ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"]
missing = [c for c in required_cols if c not in df_bronze.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df_bronze.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("payment_value")).alias("total_payment_value"),
    F.sum(F.when(F.col("payment_value") < 0, 1).otherwise(0)).alias("negative_payments")
).show()

## Section 6 — Cleaning and Standardization
1. **Standardize `order_id` & `payment_type`:** Lowercase + trim.
2. **Preserve `DECIMAL(12,2)` on `payment_value`:** Required for exact `16,008,872.12` reconciliation.
3. **Domain Rules:** `payment_sequential >= 1`, `payment_installments >= 0`, `payment_value >= 0`.

In [ ]:
df_std = (
    df_bronze
    .withColumn("order_id", F.lower(F.trim(F.col("order_id").cast("string"))))
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
    .withColumn("payment_type", F.lower(F.trim(F.col("payment_type").cast("string"))))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value", F.col("payment_value").cast("decimal(12,2)"))
)

In [ ]:
# Section 7 & 8 — Key Validation & Quarantine Design
valid_condition = (
    F.col("order_id").isNotNull() & (F.length("order_id") == 32) &
    F.col("payment_sequential").isNotNull() & (F.col("payment_sequential") >= 1) &
    F.col("payment_type").isin(VALID_PAYMENT_TYPES) &
    F.col("payment_installments").isNotNull() & (F.col("payment_installments") >= 0) &
    F.col("payment_value").isNotNull() & (F.col("payment_value") >= 0)
)

df_quarantine = (
    df_std.filter(~valid_condition)
    .withColumn("rejection_reason", F.lit("INVALID_PAYMENT_KEY_TYPE_OR_VALUE"))
    .withColumn("source_table", F.lit("dbo.order_payments"))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

df_valid = df_std.filter(valid_condition)
if df_valid.groupBy("order_id", "payment_sequential").count().filter(F.col("count") > 1).count() > 0:
    raise ValueError("Duplicate (order_id, payment_sequential) composite key detected.")

In [ ]:
# Section 9 & 10 — Silver DataFrame & Reconciliation
df_silver = (
    df_valid
    .withColumn("source_system", F.lit("sqlserver_olist"))
    .withColumn("source_table", F.lit("dbo.order_payments"))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
)

silver_count = df_silver.count()
quarantine_count = df_quarantine.count()
assert bronze_count == silver_count + quarantine_count
print(f"Reconciliation PASS: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}")

## Section 11 — Proposed Silver Write (DISABLED BY DEFAULT)
**DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.**

In [ ]:
if WRITE_ENABLED:
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET)
    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("append").saveAsTable(QUARANTINE_TARGET)
else:
    print("WRITE_ENABLED is False — dry validation only.")

In [ ]:
# Section 12 — Post-Write Validation (Future Execution Procedure Only)
# Expected: 103,886 rows, SUM(payment_value)=16008872.12